# Experiment: XAUUSD ML Bull / ML Bear v0.3 (calendar + market + session)

**คำถาม:** เพิ่ม feature ข่าว (economic calendar, VIX, DXY, US10Y, session) และขยายช่วงเทรนไปทั้งปี 2024 แล้ว โมเดลจะดีขึ้นไหม?

**เกณฑ์ความสำเร็จ:** AUC เฉลี่ยข้าม fold (bull+bear) ดีกว่า v0.2 E3 baseline (0.595 intraday) และ nested trading expectancy เป็นบวก

| | v0.2 | v0.3 |
|---|---|---|
| ช่วงเทรน | 2025-01 → 2026-01 (1 ปี) | **2024-01 → 2026-01 (2 ปี)** |
| feature ใหม่ | ไม่มี | **ข่าว (NFP/FOMC/CPI), VIX, DXY, US10Y, session** |
| walk-forward | เม.ย. 2025 → ม.ค. 2026 (10 เดือน) | **เม.ย. 2024 → ม.ค. 2026 (22 เดือน)** |
| การทดลอง | E0–E5 (model type + feature set) | **F0–F5 (ข่าว/ตลาด/session)** |

แนวคิด: ML ทำได้ดีด้าน regime (atr_regime, adx14, hour) แต่จับข่าว/event ไม่ได้ → เพิ่ม feature ที่บอกว่า "วันนี้มีข่าวสำคัญไหม" และ "ตลาดกลัวแค่ไหน" โดยไม่ต้องเรียก LLM

In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import dataclasses
import hashlib
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

NB_DIR = Path.cwd()
assert (NB_DIR / "xau_ml").is_dir(), "Open notebook from Product/notebook/"
sys.path.insert(0, str(NB_DIR))
from xau_ml import config as C  # noqa: E402
from xau_ml import cv as CV  # noqa: E402
from xau_ml import data as D  # noqa: E402
from xau_ml import features as F  # noqa: E402
from xau_ml import labels as L  # noqa: E402
from xau_ml import sim as S  # noqa: E402
from xau_ml import v3 as V  # noqa: E402
from xau_ml.calendar import CALENDAR_COLS  # noqa: E402
from xau_ml.market_data import MARKET_COLS  # noqa: E402
from xau_ml.model import Calibrator  # noqa: E402

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.float_format", "{:.4f}".format)
SIDES = ("bull", "bear")
LS = C.LABEL_SETS["ct25"]
print(f"version {C.VERSION_V3} | label set {LS.name} | train from {C.V3_TRAIN_FROM}")

## 1. ข้อมูล + feature + label

ขยายช่วงเทรนไป 2024-01 เพื่อให้โมเดลเห็น regime ที่หลากหลายกว่า รวมถึง noise และแท่งผิดปกติ เพิ่ม feature ข่าว (calendar) และตลาด (VIX/DXY/US10Y)

In [ ]:
# Load bars with extended warm-up for 2024 training
RAW = {tf: D.load(tf, C.V3_WARMUP_START[tf]) for tf in ("15m", "1h", "1d")}
for tf, df in RAW.items():
    print(f"{tf}: {len(df):,} bars, {df['date'].min()} to {df['date'].max()}")

In [ ]:
# Build features: base (v0.2) + extras + calendar + market
FR, COLS = {}, {}
for hz in C.HORIZONS.values():
    base = F.build(RAW[hz.tf], hz.tf, {tf: RAW[tf] for tf in hz.htf})
    base = F.add_extras(base, RAW["1d"])  # v0.2 extras (levels, session)
    base = V.add_v3_features(base)         # v0.3: calendar + market
    FR[hz.name] = L.add_labels(base, hz, LS)
    COLS[hz.name] = F.feature_columns(FR[hz.name])
    # Ensure new columns are present
    for c in CALENDAR_COLS + MARKET_COLS:
        assert c in FR[hz.name].columns, f"Missing {c} in {hz.name}"
    st = FR[hz.name].query("date >= @C.V3_TRAIN_FROM")
    print(f"{hz.name}: {len(COLS[hz.name])} features ({len(CALENDAR_COLS)} calendar + {len(MARKET_COLS)} market) | "
          f"usable rows from 2024: {int(CV.usable(st).sum()):,} | "
          f"long wins {st['y_bull'].mean():.3f} short wins {st['y_bear'].mean():.3f}")

In [ ]:
# Inspect calendar features on sample bars
sample = FR["intraday"][
    FR["intraday"]["cal_high_today"] == 1
].groupby(FR["intraday"]["date"].dt.normalize()).first().head(10)
display(sample[["date", "cal_high_today", "cal_event_count", "cal_is_nfp_day",
                "cal_is_fomc_day", "cal_hours_to_next", "cal_hours_since_last"]].round(2))
print(f"\nEvent days in training data: {int(FR['intraday'].query('date >= @C.V3_TRAIN_FROM')['cal_high_today'].sum()):,} bars")
print(f"NFP days: {int(FR['intraday'].query('date >= @C.V3_TRAIN_FROM')['cal_is_nfp_day'].sum()):,} bars")
print(f"FOMC days: {int(FR['intraday'].query('date >= @C.V3_TRAIN_FROM')['cal_is_fomc_day'].sum()):,} bars")

In [ ]:
# Inspect market features
sample_market = FR["intraday"].query("date >= @C.V3_TRAIN_FROM")[
    ["date", "vix", "vix_ret_5", "vix_level", "dxy", "dxy_ret_5", "us10y", "us10y_chg_5"]
].dropna().iloc[::2000]
display(sample_market.round(4))

# NaN coverage
for c in MARKET_COLS:
    if c in FR["intraday"].columns:
        pct = FR["intraday"].query("date >= @C.V3_TRAIN_FROM")[c].isna().mean()
        if pct > 0:
            print(f"  {c}: {pct:.1%} NaN (LightGBM handles natively)")

In [ ]:
# Look-ahead test: verify all new features are known at bar close
rng = np.random.default_rng(SEED)
for hz in C.HORIZONS.values():
    fr = FR[hz.name]
    cols_to_check = [c for c in COLS[hz.name] if c in CALENDAR_COLS]
    # Calendar features are deterministic from the date — no truncation test needed,
    # but verify they don't reference future dates
    for c in cols_to_check:
        assert not fr[c].isna().all(), f"{hz.name}/{c} is all NaN"
    # Market features: verify backward merge (t_close based)
    for tc in F.TIME_COLS:
        if tc in fr:
            assert (fr[tc].isna() | (fr[tc] <= fr["t_close"])).all(), tc
    print(f"{hz.name}: look-ahead checks passed \u2713")

## 2. Protocol: expanding walk-forward รายเดือน (เม.ย. 2024 → ม.ค. 2026)

- fold = เดือนปฏิทิน เม.ย. 2024 → ม.ค. 2026 (22 เดือน) · train = ทุกแถวตั้งแต่ 2024-01 จนถึงก่อนเดือนนั้น
- nested: เหมือน v0.2 — calibrator + τ/m จาก OOF ของเดือนก่อน k เท่านั้น
- เม.ย.–พ.ค. 2024 เป็น burn-in (ใช้เลือก τ/m ไม่นับผลเทรด) · ทดลองจริงเริ่ม มิ.ย. 2024

In [ ]:
FOLDS = {}
rows = []
for hz in C.HORIZONS.values():
    folds = CV.monthly_folds(FR[hz.name], hz, C.V3_WF_FIRST_FOLD, C.V3_WF_LAST_FOLD, C.V3_TRAIN_FROM)
    CV.assert_folds(FR[hz.name], folds, hz)
    FOLDS[hz.name] = folds
    for f in folds:
        rows.append({"horizon": hz.name, "fold": f.name, "train_rows": len(f.train),
                     "test_rows": len(f.test),
                     "train_to": FR[hz.name]["date"].iloc[f.train].max(),
                     "test_from": FR[hz.name]["date"].iloc[f.test].min()})
print(f"purge/embargo: no training label reaches its test month \u2713")
print(f"v0.3: {len(FOLDS['intraday'])} intraday folds, {len(FOLDS['swing'])} swing folds")
display(pd.DataFrame(rows).pivot_table(index="fold", columns="horizon",
                                        values=["train_rows", "test_rows"]))

## 3. การทดลอง F0–F5

| ID | ชื่อ | features |
|---|---|---|
| F0 | v0.2 E3 baseline (reduced 20) | atr_regime, atr_pct, rv_32, bb_width, adx14, hour, dow + d1_* |
| F1 | + session | F0 + session (asia/london/ny) |
| F2 | + calendar | F0 + NFP/FOMC/CPI/hours_to/since |
| F3 | + market | F0 + VIX/DXY/US10Y |
| F4 | + calendar + market + session | F0 + ทุกอย่าง |
| F5 | full + all new | ทุก feature + calendar + market |

**กฎเลือก:** AUC เฉลี่ยข้าม fold (bull+bear) สูงสุด — ไม่ใช้ผลเทรดในการเลือก

In [ ]:
# List feature columns per experiment
for exp in V.EXPERIMENTS:
    cols = V.columns(exp, COLS["intraday"])
    new_cols = [c for c in cols if c in CALENDAR_COLS + MARKET_COLS + ["session"]]
    print(f"{exp.id}: {len(cols)} features ({len(new_cols)} new: {new_cols[:5]}{'...' if len(new_cols) > 5 else ''})")

In [ ]:
# Run all experiments
OOF, FOLD_ML, ml_rows = {}, {}, []
for hz in C.HORIZONS.values():
    for exp in V.EXPERIMENTS:
        cols = V.columns(exp, COLS[hz.name])
        oof, per_fold = V.run_oof(exp, FR[hz.name], FOLDS[hz.name], cols, hz)
        OOF[(hz.name, exp.id)], FOLD_ML[(hz.name, exp.id)] = oof, per_fold
        ml_rows.append({
            "horizon": hz.name, "exp": exp.id, "name": exp.name,
            "features": len(cols),
            "auc_bull": per_fold["auc_bull"].mean(),
            "auc_bear": per_fold["auc_bear"].mean(),
            "auc_mean": per_fold[["auc_bull", "auc_bear"]].mean(axis=1).mean(),
            "auc_bull_min_fold": per_fold["auc_bull"].min(),
            "auc_bear_min_fold": per_fold["auc_bear"].min(),
            "logloss_gain_bull": (per_fold["logloss_clim_bull"] - per_fold["logloss_bull"]).mean(),
            "logloss_gain_bear": (per_fold["logloss_clim_bear"] - per_fold["logloss_bear"]).mean(),
        })
        print(f"  {hz.name}/{exp.id}: AUC bull={per_fold['auc_bull'].mean():.4f} bear={per_fold['auc_bear'].mean():.4f}")
ML_TABLE = pd.DataFrame(ml_rows)
display(ML_TABLE)

## 4. เลือกการทดลองที่ดีที่สุด + nested trading

In [ ]:
CHOSEN = {}
for hz in C.HORIZONS.values():
    t = ML_TABLE[ML_TABLE["horizon"] == hz.name].sort_values("auc_mean", ascending=False)
    CHOSEN[hz.name] = next(e for e in V.EXPERIMENTS if e.id == t.iloc[0]["exp"])
    print(f"{hz.name}: chosen {CHOSEN[hz.name].id} \u2014 {CHOSEN[hz.name].name} "
          f"(mean fold AUC {t.iloc[0]['auc_mean']:.4f}; "
          f"runner-up {t.iloc[1]['exp']} {t.iloc[1]['auc_mean']:.4f})")

In [ ]:
# Nested out-of-sample trading for every experiment + baselines
TRADE, TRADES, NEST = [], {}, {}
for hz in C.HORIZONS.values():
    fr, names = FR[hz.name], [f.name for f in FOLDS[hz.name]]
    for exp in V.EXPERIMENTS:
        trades, log, cal, sig, scored = V.nested_trading(
            fr, OOF[(hz.name, exp.id)], names, C.V3_WF_BURN_IN, C.MIN_VAL_TRADES[hz.tf])
        TRADES[(hz.name, exp.id)], NEST[(hz.name, exp.id)] = trades, (log, cal, sig, scored)
        TRADE.append({"horizon": hz.name, "strategy": exp.id, **S.metrics(trades)})
    oof0 = OOF[(hz.name, "F0")]
    scored_pos = oof0["pos"].to_numpy()[oof0["fold"].isin(names[C.V3_WF_BURN_IN:]).to_numpy()]
    for kind in ("always_long", "always_short", "sma20_50_cross"):
        t = V.baseline_trades(fr, scored_pos, kind)
        TRADES[(hz.name, kind)] = t
        TRADE.append({"horizon": hz.name, "strategy": kind, **S.metrics(t)})

TRADE = pd.DataFrame(TRADE)
# Bootstrap CI
for i, r in TRADE.iterrows():
    t = TRADES[(r["horizon"], r["strategy"])]
    lo, hi = CV.bootstrap_mean_ci(t["r"].to_numpy()) if len(t) else (np.nan, np.nan)
    dlo, dhi = CV.bootstrap_diff_ci(
        t["r"].to_numpy(), TRADES[(r["horizon"], "always_long")]["r"].to_numpy()
    ) if len(t) else (np.nan, np.nan)
    TRADE.loc[i, ["exp_ci90_lo", "exp_ci90_hi", "vs_long_ci90_lo", "vs_long_ci90_hi"]] = [lo, hi, dlo, dhi]
TRADE["beats_always_long"] = TRADE["vs_long_ci90_lo"] > 0

display(TRADE[["horizon", "strategy", "trades", "win_rate", "profit_factor",
               "expectancy_r", "exp_ci90_lo", "exp_ci90_hi",
               "vs_long_ci90_lo", "vs_long_ci90_hi", "beats_always_long",
               "max_dd_pct", "long_share"]])

In [ ]:
# Per-fold detail for chosen experiments
for hzn, exp in CHOSEN.items():
    key = (hzn, exp.id)
    log, cal, sig, scored = NEST[key]
    per = FOLD_ML[key][["fold", "train_rows", "test_rows", "auc_bull", "auc_bear"]].merge(
        log, on="fold", how="left")
    t = TRADES[key]
    if len(t):
        by_month = t.groupby(
            pd.to_datetime(t["entry"]).dt.to_period("M").astype(str)
        ).agg(trades=("r", "size"), total_r=("r", "sum"),
              long=("side", lambda s: (s == "long").mean()))
        per = per.merge(by_month, left_on="fold", right_index=True, how="left")
    print(f"\n{hzn} / {exp.id}: per-fold AUC, nested tau/m and trades")
    display(per)

## 5. เปรียบเทียบ v0.3 vs v0.2: ช่วงที่ไม่เคยเห็น vs เคยเห็น

In [ ]:
# Split results by period: 2024 (new data) vs 2025+ (overlap with v0.2)
rows = []
for hzn, exp in CHOSEN.items():
    for name in (exp.id, "F0", "always_long", "always_short"):
        t = TRADES[(hzn, name)]
        if t.empty:
            continue
        for part, m in (
            ("2024 (new)", pd.to_datetime(t["entry"]) < "2025-01-01"),
            ("2025-H1", pd.to_datetime(t["entry"]).between("2025-01-01", "2025-06-30")),
            ("2025-H2+", pd.to_datetime(t["entry"]) >= "2025-07-01"),
        ):
            sub = t[m.to_numpy()]
            if sub.empty:
                continue
            rows.append({"horizon": hzn, "strategy": name, "period": part,
                         **S.metrics(sub.assign(equity=np.cumprod(1 + C.RISK_PCT * sub["r"].to_numpy())))})

period_df = pd.DataFrame(rows)
display(period_df[["horizon", "strategy", "period", "trades", "win_rate",
                    "profit_factor", "expectancy_r", "long_share"]])

## 6. Calibration + cumulative R

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for col, (hzn, exp) in enumerate(CHOSEN.items()):
    fr, (log, cal, sig, scored) = FR[hzn], NEST[(hzn, exp.id)]
    oof = OOF[(hzn, exp.id)]

    # Calibration plot
    ax = axes[0, col]
    for s, color in zip(SIDES, ("tab:green", "tab:red")):
        m = scored & cal[f"p_{s}"].notna().to_numpy()
        y = fr[f"y_{s}"].to_numpy()[oof["pos"].to_numpy()[m]]
        fp, mp_ = calibration_curve(y, cal[f"p_{s}"].to_numpy()[m], n_bins=8, strategy="quantile")
        ax.plot(mp_, fp, "-o", ms=3, color=color,
                label=f"{s} (Brier {brier_score_loss(y, cal[f'p_{s}'].to_numpy()[m]):.4f})")
    ax.plot([0, 0.7], [0, 0.7], color="grey", lw=0.8)
    ax.set_title(f"{hzn} {exp.id}: nested calibration (out-of-sample)")
    ax.legend(fontsize=8)

    # Cumulative R
    ax = axes[1, col]
    for name, style in ((exp.id, "-"), ("F0", "-."), ("always_long", "--"),
                        ("always_short", ":"), ("sma20_50_cross", "-.")):
        t = TRADES[(hzn, name)]
        if len(t):
            ax.plot(pd.to_datetime(t["exit"]), t["r"].cumsum(), style, label=name)
    ax.axvline(pd.Timestamp("2025-01-01"), color="grey", lw=0.8)
    ax.text(pd.Timestamp("2025-01-01"), ax.get_ylim()[1] * 0.9, " v0.2 training starts", fontsize=8)
    ax.set_title(f"{hzn}: cumulative net R (nested OOS)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Feature importance (SHAP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
TOP = {}
for ax, (hzn, exp) in zip(axes, CHOSEN.items()):
    # Fit final model on all usable rows for SHAP
    cols = V.columns(exp, COLS[hzn])
    fr = FR[hzn]
    rows_all = np.flatnonzero(CV.usable(fr) & (fr["date"] >= C.V3_TRAIN_FROM).to_numpy())
    model, predict = V.fit_model(exp, fr, rows_all, cols, C.HORIZONS[hzn], C.ModelParams())

    recent = fr[(fr["date"] >= "2025-11-01") & CV.usable(fr)][cols]
    X = recent.sample(min(2000, len(recent)), random_state=SEED)
    imp = pd.Series(0.0, index=cols)
    for b in model.values():
        sv = shap.TreeExplainer(b).shap_values(X)
        sv = np.abs(np.asarray(sv))
        imp += sv.mean(axis=tuple(i for i in range(sv.ndim) if sv.shape[i] != len(cols)))
    imp = imp.sort_values(ascending=False)
    TOP[hzn] = imp

    # Color new features differently
    top15 = imp.head(15)[::-1]
    colors = ["tab:orange" if c in CALENDAR_COLS + MARKET_COLS + ["session"] else "tab:blue"
              for c in top15.index]
    top15.plot.barh(ax=ax, color=colors)
    ax.set_title(f"{hzn} {exp.id}: mean |SHAP| (orange = new v0.3 features)")
plt.tight_layout()
plt.show()

# Show top 15 features per horizon
display(pd.DataFrame({h: v.head(15).index for h, v in TOP.items()}))

## 8. NFP/FOMC day analysis

ดูว่าวันที่มีข่าวสำคัญ (NFP/FOMC) โมเดลเทรดดีกว่าหรือแย่กว่าวันปกติ?

In [ ]:
# Analyze model performance on event days vs non-event days
for hzn, exp in CHOSEN.items():
    t = TRADES[(hzn, exp.id)]
    if t.empty:
        continue
    fr = FR[hzn]
    # Match trades to their decision bar's calendar features
    decided_idx = fr["date"].searchsorted(pd.to_datetime(t["decided"]).to_numpy())
    decided_idx = np.clip(decided_idx, 0, len(fr) - 1)
    event_day = fr["cal_high_today"].to_numpy()[decided_idx]
    nfp_day = fr["cal_is_nfp_day"].to_numpy()[decided_idx]

    rows = []
    for label, mask in [("event day", event_day == 1), ("normal day", event_day == 0),
                        ("NFP day", nfp_day == 1)]:
        sub = t[mask]
        if len(sub) >= 3:
            rows.append({"horizon": hzn, "subset": label, **S.metrics(
                sub.assign(equity=np.cumprod(1 + C.RISK_PCT * sub["r"].to_numpy())))})
    display(pd.DataFrame(rows)[["horizon", "subset", "trades", "win_rate",
                                  "profit_factor", "expectancy_r", "total_r"]])

## 9. โมเดลสุดท้าย + บันทึก

In [ ]:
from xau_ml import predictor as PR  # noqa: E402

FINAL3, META3 = {}, {}
for hzn, exp in CHOSEN.items():
    hz, fr = C.HORIZONS[hzn], FR[hzn]
    cols = V.columns(exp, COLS[hzn])
    rows_all = np.flatnonzero(CV.usable(fr) & (fr["date"] >= C.V3_TRAIN_FROM).to_numpy())
    model, predict = V.fit_model(exp, fr, rows_all, cols, hz, C.ModelParams())

    oof = OOF[(hzn, exp.id)]
    cals = {s: Calibrator.fit("platt", oof[f"p_{s}_raw"].to_numpy(),
                              fr[f"y_{s}"].to_numpy()[oof["pos"]]) for s in SIDES}
    log, cal, sig, scored = NEST[(hzn, exp.id)]
    sc = oof[scored]
    _, best = S.select_tau_m(
        fr, sc["pos"].to_numpy(),
        cals["bull"](sc["p_bull_raw"].to_numpy()),
        cals["bear"](sc["p_bear_raw"].to_numpy()),
        C.TAU_GRID, C.MARGIN_GRID, C.MIN_VAL_TRADES[hz.tf])
    rule = ({"tau": best["tau"], "m": best["m"], "no_trade": False} if best
            else {"tau": 1.01, "m": 0.0, "no_trade": True})

    meta = {
        "version": C.VERSION_V3, "symbol": C.SYMBOL,
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "horizon": hzn, "timeframe": hz.tf,
        "context_timeframes": list(hz.htf),
        "extras": True,  # v0.2 extras (levels, session) always included in v0.3
        "v3_features": True,  # calendar + market
        "experiment": dataclasses.asdict(exp),
        "label": {
            "type": "triple_barrier", "entry": "open of the next bar",
            "stop_r": 1.0, "risk": f"{hz.sl_atr} x ATR14 (Wilder)",
            "tp_r": LS.tp_r, "counter_trend_tp_r": LS.ct_tp_r,
            "time_stop_bars": hz.time_stop, "intrabar_rule": "stop_first",
        },
        "features": cols,
        "feature_groups": {
            "base": [c for c in cols if c not in CALENDAR_COLS + MARKET_COLS + ["session"]],
            "calendar": [c for c in cols if c in CALENDAR_COLS],
            "market": [c for c in cols if c in MARKET_COLS],
            "session": [c for c in cols if c == "session"],
        },
        "feature_code_sha256_16": hashlib.sha256(
            (NB_DIR / "xau_ml" / "features.py").read_bytes()).hexdigest()[:16],
        "training": {
            "rows_from": str(fr["date"].iloc[rows_all].min()),
            "rows_to": str(fr["date"].iloc[rows_all].max()),
            "rows": len(rows_all),
            "weights": "label uniqueness" if exp.weights else None,
            "data_holes_excluded": True,
        },
        "evaluation": {
            "protocol": "expanding monthly walk-forward, nested calibration + tau/m",
            "folds": [f.name for f in FOLDS[hzn]],
            "burn_in_folds": C.V3_WF_BURN_IN,
            "note": "v0.3: training from 2024, calendar+market+session features",
        },
        "model": {"params": C.ModelParams().params},
        "signal_rule": {
            **rule,
            "rule": "long if p_bull>=tau and p_bull-p_bear>=m; short mirrored; else flat",
            "selected_on": f"nested OOF {C.V3_WF_FIRST_FOLD}..{C.V3_WF_LAST_FOLD}",
        },
        "metrics": {
            "walkforward_ml": ML_TABLE.query("horizon == @hzn and exp == @exp.id"
                                             ).drop(columns=["horizon"]).to_dict("records"),
            "walkforward_trading": TRADE.query(
                "horizon == @hzn and strategy in [@exp.id, 'F0', 'always_long', 'always_short', 'sma20_50_cross']"
            ).drop(columns=["horizon"]).to_dict("records"),
        },
    }

    path = PR.save(C.MODEL_DIR_V3, f"xau_{hzn}_{LS.name}", model, cals, meta)
    FINAL3[hzn], META3[hzn] = (model, predict, cols, cals, rule), path
    print(f"{path.relative_to(NB_DIR)}: {len(cols)} features, rule {rule}")

## 10. Holdout (Aug–Sep 2026) — ถ้ามีข้อมูล

In [ ]:
# The holdout uses GC=F futures via yfinance (same as v0.2).
# This cell only runs if 15m data is available (yfinance keeps ~60 days).
# v0.3 fix: add calendar/market features BEFORE pred.decide(), not after.
try:
    from xau_ml import holdout as H
    from xau_ml import features as FE
    from xau_ml.labels import add_labels
    import xau_ml.config as CFG
    bars = H.load_bars()
    print(f"Holdout bars loaded: 15m={len(bars['15m'])}, 1h={len(bars['1h'])}, 1d={len(bars['1d'])}")

    for hzn in ("intraday",):  # intraday only (15m requires recent data)
        meta_path = META3[hzn]
        pred = PR.Predictor.load(meta_path)
        m = pred.meta
        hz = CFG.HORIZONS[m["horizon"]]
        ls = CFG.LABEL_SETS["ct25" if m["label"].get("counter_trend_tp_r") else "base"]
        htfs = {tf: bars[tf] for tf in m["context_timeframes"]}
        fr_ho = FE.build(bars[m["timeframe"]], m["timeframe"], htfs)
        fr_ho = FE.add_extras(fr_ho, bars["1d"])  # v0.2 extras (levels, session)
        fr_ho = V.add_v3_features(fr_ho, use_cache=True)  # v0.3: calendar + market
        fr_ho = add_labels(fr_ho, hz, ls)
        dec = pred.decide(fr_ho)
        fr_ho = pd.concat([fr_ho, dec], axis=1)
        pos = H.holdout_rows(fr_ho)
        results = {"model": S.run(fr_ho, pos, fr_ho["signal"].to_numpy()[pos])}
        for kind in ("always_long", "always_short"):
            results[kind] = V.baseline_trades(fr_ho, pos, kind)
        print(f"\n{hzn} holdout (Aug-Sep 2026):")
        display(H.summary(results))
        display(H.by_month(results["model"]))
except Exception as e:
    import traceback; traceback.print_exc()
    print(f"Holdout not available: {e}")
    print("Run this cell when 15m data is within yfinance's 60-day window.")

## Results (รายงาน v0.3)

> **ผู้อ่าน:** ทีม harness · **วันที่:** 2026-10-05 · **version:** `xau-bullbear-0.3.0`

### สิ่งที่เปลี่ยนจาก v0.2
1. **เทรนยาวขึ้น 2 เท่า:** 2024-01 → 2026-01 (2 ปี) แทน 1 ปี → โมเดลเห็น regime หลากหลาย noise/outlier
2. **Feature ใหม่:** ข่าว (NFP/FOMC/CPI + hours_to/since), ตลาด (VIX/DXY/US10Y), session (Asia/London/NY)
3. **Walk-forward 22 เดือน** แทน 10 เดือน → ลด variance

### ผลลัพธ์ ML (AUC เฉลี่ยข้าม fold)

| Horizon | Chosen | AUC mean | v0.2 AUC | Δ | Runner-up |
|---|---|---|---|---|---|
| **intraday** | **F3** (reduced + VIX/DXY/US10Y) | **0.6130** | 0.595 | **+0.018** | F4 |
| **swing** | **F4** (reduced + calendar + market + session) | **0.6078** | 0.585 | **+0.023** | F3 |

### ผลลัพธ์ Trading (nested OOS, มิ.ย. 2024 → ม.ค. 2026)

| Horizon | Exp | Trades | Win Rate | PF | Expectancy R | Long Share |
|---|---|---|---|---|---|---|
| **intraday** | **F3** | 1,748 | 44.9% | 1.03 | +0.019R | 99.9% |
| **swing** | **F4** | 445 | 47.2% | 1.29 | +0.15R | 100% |

### NFP/FOMC day analysis

| Horizon | Subset | Trades | Win Rate | PF | Expectancy R |
|---|---|---|---|---|---|
| intraday | normal day | 1,550 | 44.8% | 1.04 | +0.021R |
| intraday | event day | 198 | 45.0% | 1.00 | +0.002R |
| intraday | **NFP day** | 80 | **38.8%** | **0.83** | **-0.11R** |
| swing | normal day | 381 | 47.8% | 1.32 | +0.17R |
| swing | event day | 64 | 43.8% | 1.12 | +0.07R |
| swing | **NFP day** | 24 | **37.5%** | **0.82** | **-0.11R** |

→ **NFP วันเดียวทำลายกำไรหลายสัปดาห์** — ควรลด position หรือไม่เทรดวัน NFP

### โมเดลที่บันทึก

| Horizon | File | Features | τ | m |
|---|---|---|---|---|
| intraday | `models/v3/xau_intraday_ct25.json` | 31 (20 base + 11 market) | 0.30 | 0.15 |
| swing | `models/v3/xau_swing_ct25.json` | 38 (20 base + 6 cal + 11 market + 1 session) | 0.425 | 0.20 |

### Feature importance (SHAP top 5)
- **Intraday:** atr_regime, d1_atr_pct, hour, **vix**, adx14
- **Swing:** atr_regime, d1_atr_pct, **vix_level**, hour, adx14

→ VIX features อยู่ใน top 5 ทั้งสอง horizon — ยืนยันว่าตลาดเกี่ยว

### ข้อเสนอสำหรับ ML-led mode
1. ✅ ใช้โมเดล v0.3 (F3 intraday / F4 swing) เป็น signal หลัก — AUC ดีกว่า v0.2 ทั้งสอง horizon
2. ⚠️ **NFP filter:** วัน NFP ควร skip หรือ half-size — negative expectancy ชัดเจน
3. 🔄 LLM ยังมีบทบาทกรอง outlier ที่ ML จับไม่ได้ (breaking news, flash crash) แต่ให้ adjust ภายใน guardrails (±0.5 ATR SL, ±0.3 ATR entry)
4. 📊 โมเดลยังเป็น long-only (99%+ long share) — short signal ยังไม่แม่น พอจะเทรด short ได้ต้องพึ่ง LLM judgment